<a href="https://colab.research.google.com/github/BF667/notebook/blob/main/RVC_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC Trainer Notebook
This notebook provides a complete pipeline for training Retrieval-based Voice Conversion (RVC) models.

## Setup Instructions
1. Run cells in order
2. Ensure GPU is enabled (Runtime → Change runtime type → GPU)
3. Mount Google Drive for persistent storage (recommended)

In [ ]:
# @title Mount Google Drive (Optional)
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    print('Google Drive mounted successfully')
except Exception as e:
    print(f'Error mounting Google Drive: {e}')

# Create backup directory if it doesn't exist
BACKUP_PATH = '/content/drive/MyDrive/RVC_Backup'
os.makedirs(BACKUP_PATH, exist_ok=True)

In [ ]:
# @title Setup Runtime Environment
from multiprocessing import cpu_count
import os
import sys
import subprocess

# Constants
REPO_URL = 'https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git'
BASE_DIR = '/content/RVC'
REQUIREMENTS_FILE = 'requirements.txt'

# Global variables
cpu_cores = cpu_count()
hop_length = 128
now_dir = os.getcwd()

def run_command(command, error_message):
    try:
        result = subprocess.run(command, shell=True, check=True, text=True, capture_output=True)
        print(result.stdout)
        return True
    except subprocess.CalledProcessError as e:
        print(f'{error_message}: {e.stderr}')
        return False

# Clone repository
if not os.path.exists(BASE_DIR):
    print('Cloning RVC repository...')
    run_command(f'git clone {REPO_URL} {BASE_DIR}', 'Failed to clone repository')

# Change to RVC directory
os.chdir(BASE_DIR)

# Set up Python environment
print('Setting up Python environment...')
run_command('sudo update-alternatives --set python3 /usr/bin/python3.10', 'Failed to set Python version')
run_command('curl -LsSf https://astral.sh/uv/install.sh | sh', 'Failed to install uv')

# Install requirements
print('Installing requirements...')
if run_command(f'uv pip install -q -r {REQUIREMENTS_FILE}', 'Failed to install requirements'):
    print('Requirements installed successfully')

# Download prerequisites
run_command('python core.py prerequisites --models True --pretraineds_hifigan True', 'Failed to download prerequisites')

sys.path.append(BASE_DIR)

In [ ]:
# @title Download Audio
from rvc.lib.tools.yt import download_audio
import re

def validate_url(url):
    pattern = r'^(https?:\/\/)?((w{3}\.)?youtube\.com|youtu\.be)\/.*$'
    return re.match(pattern, url) is not None

url = 'https://youtu.be/_bFFnDrlhF4?si=fPFCYV-YczHka_Zk'  # @param {type:"string"}
output_dir = '/content/RVC/dataset'

if not validate_url(url):
    print('Invalid YouTube URL. Please provide a valid YouTube link.')
else:
    os.makedirs(output_dir, exist_ok=True)
    try:
        download_audio(urls=[url], output_dir=output_dir)
        print(f'Audio downloaded successfully to {output_dir}')
    except Exception as e:
        print(f'Error downloading audio: {e}')

In [ ]:
# @title Dataset Maker
from rvc.lib.tools.dataset import split_audio_files
import glob

dataset_dir = '/content/RVC/dataset'  # @param {type:"string"}

if not os.path.exists(dataset_dir):
    print(f'Dataset directory {dataset_dir} does not exist')
else:
    audio_files = glob.glob(f'{dataset_dir}/*.wav')
    if not audio_files:
        print('No WAV files found in dataset directory')
    else:
        try:
            split_audio_files(audio_files)
            print('Dataset processing completed')
        except Exception as e:
            print(f'Error processing dataset: {e}')

In [ ]:
# @title Setup Model Variables
model_name = 'Darwin'  # @param {type:"string"}
sample_rate = '40k'  # @param ['32k', '40k', '48k'] {allow-input: false}

if not model_name.strip():
    raise ValueError('Model name cannot be empty')

sr = int(sample_rate.rstrip('k')) * 1000
print(f'Model {model_name} configured with sample rate {sample_rate}')

In [ ]:
# @title Preprocess Dataset
from os import environ
import os

environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
dataset_path = '/content/drive/MyDrive/Darwin_Dataset'  # @param {type:"string"}
cut_preprocess = 'Automatic'  # @param ['Skip', 'Simple', 'Automatic']
chunk_len = 3  # @param {type:"slider", min:0.5, max:5.0, step:0.5}
overlap_len = 0.3  # @param {type:"slider", min:0, max:0.5, step:0.1}
process_effects = False  # @param {type:"boolean"}
noise_reduction = False  # @param {type:"boolean"}
noise_reduction_strength = 0.7  # @param {type:"slider", min:0.0, max:1.0, step:0.1}

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f'Dataset path {dataset_path} does not exist')

os.chdir(BASE_DIR)
command = (f'python core.py preprocess --model_name "{model_name}" --dataset_path "{dataset_path}" '
           f'--sample_rate "{sr}" --cpu_cores "{cpu_cores}" --cut_preprocess "{cut_preprocess}" '
           f'--process_effects "{process_effects}" --noise_reduction "{noise_reduction}" '
           f'--noise_reduction_strength "{noise_reduction_strength}" --chunk_len "{chunk_len}" '
           f'--overlap_len "{overlap_len}"')
run_command(command, 'Failed to preprocess dataset')

In [ ]:
# @title Extract Features
f0_method = 'rmvpe'  # @param ['crepe', 'crepe-tiny', 'rmvpe'] {allow-input: false}
include_mutes = 2  # @param {type:"slider", min:0, max:10, step:1}
embedder_model = 'contentvec'  # @param ['contentvec', 'chinese-hubert-base', 'japanese-hubert-base', 'korean-hubert-base', 'custom'] {allow-input: false}
embedder_model_custom = ''  # @param {type:"string"}

os.chdir(BASE_DIR)
command = (f'python core.py extract --model_name "{model_name}" --f0_method "{f0_method}" '
           f'--hop_length "{hop_length}" --sample_rate "{sr}" --cpu_cores "{cpu_cores}" '
           f'--gpu "0" --embedder_model "{embedder_model}" --embedder_model_custom "{embedder_model_custom}" '
           f'--include_mutes "{include_mutes}"')
run_command(command, 'Failed to extract features')

In [ ]:
# @title Generate Index File
index_algorithm = 'Auto'  # @param ['Auto', 'Faiss', 'KMeans'] {allow-input: false}

os.chdir(BASE_DIR)
run_command(f'python core.py index --model_name "{model_name}" --index_algorithm "{index_algorithm}"', 
           'Failed to generate index file')

In [ ]:
# @title Start Training
import threading
import time
import os
import shutil
from datetime import datetime

LOGS_FOLDER = f'{BASE_DIR}/logs'
BACKUP_PATH = '/content/drive/MyDrive/RVC_Backup'

def backup_files():
    print('Starting backup loop...')
    last_backup_timestamps_path = os.path.join(LOGS_FOLDER, 'last_backup_timestamps.txt')
    fully_updated = False

    while True:
        try:
            stats = {'new': 0, 'updated': 0, 'deleted': 0}
            last_backup_timestamps = {}

            try:
                with open(last_backup_timestamps_path, 'r') as f:
                    last_backup_timestamps = dict(line.strip().split(':', 1) for line in f if ':' in line)
            except FileNotFoundError:
                pass

            for root, dirs, files in os.walk(LOGS_FOLDER):
                dirs[:] = [d for d in dirs if d not in ('zips', 'mute')]
                for filename in files:
                    if filename != 'last_backup_timestamps.txt':
                        filepath = os.path.join(root, filename)
                        backup_filepath = os.path.join(BACKUP_PATH, os.path.relpath(filepath, LOGS_FOLDER))
                        os.makedirs(os.path.dirname(backup_filepath), exist_ok=True)
                        
                        current_timestamp = str(os.path.getmtime(filepath))
                        last_timestamp = last_backup_timestamps.get(filepath)
                        
                        if last_timestamp is None or float(last_timestamp) < float(current_timestamp):
                            shutil.copy2(filepath, backup_filepath)
                            last_backup_timestamps[filepath] = current_timestamp
                            stats['new' if last_timestamp is None else 'updated'] += 1

            for filepath in list(last_backup_timestamps.keys()):
                if not os.path.exists(filepath):
                    backup_filepath = os.path.join(BACKUP_PATH, os.path.relpath(filepath, LOGS_FOLDER))
                    if os.path.exists(backup_filepath):
                        os.remove(backup_filepath)
                        stats['deleted'] += 1
                    del last_backup_timestamps[filepath]

            if any(stats.values()):
                print(f'Backup Complete: {stats["new"]} new, {stats["updated"]} updated, {stats["deleted"]} deleted')
                fully_updated = False
            elif not fully_updated:
                print('Files are up to date')
                fully_updated = True

            with open(last_backup_timestamps_path, 'w') as f:
                for filepath, timestamp in last_backup_timestamps.items():
                    f.write(f'{filepath}:{timestamp}\n')

            time.sleep(cooldown if fully_updated else 0.1)

        except Exception as error:
            print(f'Backup error: {error}')

# Training parameters
total_epoch = 800  # @param {type:"integer"}
batch_size = 8  # @param {type:"slider", min:1, max:25, step:1}
gpu = 0
pretrained = True  # @param {type:"boolean"}
cleanup = False  # @param {type:"boolean"}
cache_data_in_gpu = False  # @param {type:"boolean"}
vocoder = 'HiFi-GAN'  # @param ['HiFi-GAN']
checkpointing = False
tensorboard = True  # @param {type:"boolean"}
save_every_epoch = 10  # @param {type:"slider", min:1, max:100, step:1}
save_only_latest = False  # @param {type:"boolean"}
save_every_weights = False  # @param {type:"boolean"}
overtraining_detector = False  # @param {type:"boolean"}
overtraining_threshold = 50  # @param {type:"slider", min:1, max:100, step:1}
custom_pretrained = False  # @param {type:"boolean"}
g_pretrained_path = f'{BASE_DIR}/rvc/models/pretraineds/pretraineds_custom/G48k.pth'  # @param {type:"string"}
d_pretrained_path = f'{BASE_DIR}/rvc/models/pretraineds/pretraineds_custom/D48k.pth'  # @param {type:"string"}
auto_backups = True  # @param {type:"boolean"}
cooldown = 15  # @param {type:"slider", min:0, max:100, step:1}

def start_train():
    if tensorboard:
        %load_ext tensorboard
        %tensorboard --logdir {LOGS_FOLDER}

    os.chdir(BASE_DIR)
    command = (f'python core.py train --model_name "{model_name}" --save_every_epoch "{save_every_epoch}" '
               f'--save_only_latest "{save_only_latest}" --save_every_weights "{save_every_weights}" '
               f'--total_epoch "{total_epoch}" --sample_rate "{sr}" --batch_size "{batch_size}" '
               f'--gpu "{gpu}" --pretrained "{pretrained}" --custom_pretrained "{custom_pretrained}" '
               f'--g_pretrained_path "{g_pretrained_path}" --d_pretrained_path "{d_pretrained_path}" '
               f'--overtraining_detector "{overtraining_detector}" --overtraining_threshold "{overtraining_threshold}" '
               f'--cleanup "{cleanup}" --cache_data_in_gpu "{cache_data_in_gpu}" --vocoder "{vocoder}" '
               f'--checkpointing "{checkpointing}"')
    run_command(command, 'Failed to start training')

if auto_backups:
    server_thread = threading.Thread(target=start_train)
    server_thread.start()
    backup_files()
else:
    start_train()

In [ ]:
# @title Export Model
from pathlib import Path
import subprocess

export_for = 'training'  # @param ['training', 'inference'] {allow-input: false}
logs_folder = Path(f'{BASE_DIR}/logs/{model_name}')

if not (logs_folder.exists() and logs_folder.is_dir()):
    raise FileNotFoundError(f'Model folder {model_name} not found')

os.chdir(logs_folder.parent)
output_zip = f'/content/{model_name}.zip'

if export_for == 'training':
    run_command(f'zip -r "{output_zip}" "{model_name}"', 'Failed to create training zip')
else:
    weight_files = sorted(logs_folder.glob(f'{model_name}/*e_*s.pth'), key=os.path.getmtime, reverse=True)
    if not weight_files:
        raise FileNotFoundError('No weight file found. Please complete training first.')
    
    weight_path = weight_files[0]
    index_path = logs_folder / f'{model_name}.index'
    files_to_zip = [str(weight_path)]
    if index_path.exists():
        files_to_zip.append(str(index_path))
    
    run_command(f'zip "{output_zip}" {" ".join(files_to_zip)}', 'Failed to create inference zip')

if Path('/content/drive').is_mount():
    os.makedirs(BACKUP_PATH, exist_ok=True)
    shutil.move(output_zip, BACKUP_PATH)
    print(f'Exported model as {BACKUP_PATH}/{model_name}.zip')
else:
    print(f'Drive not mounted, exported model as {output_zip}')

In [ ]:
# @title Load Backup
import os
import shutil

model_name = 'Darwin'  # @param {type:"string"}
source_path = f'/content/drive/MyDrive/RVC_Backup/{model_name}'
destination_path = f'{BASE_DIR}/logs/{model_name}'
backup_timestamps_file = 'last_backup_timestamps.txt'

if not os.path.exists(source_path):
    print(f'Model folder {source_path} does not exist. Please verify the name or check Google Drive.')
else:
    try:
        time_source = os.path.join('/content/drive/MyDrive/RVC_Backup', backup_timestamps_file)
        time_dest = os.path.join(f'{BASE_DIR}/logs', backup_timestamps_file)
        if os.path.exists(time_source):
            shutil.copy2(time_source, time_dest)
        
        if os.path.exists(destination_path):
            shutil.rmtree(destination_path)
        shutil.copytree(source_path, destination_path)
        print('Model backup loaded successfully')
    except Exception as e:
        print(f'Error loading backup: {e}')